# Animal Image Classification Project

This notebook demonstrates a comprehensive Computer Vision and Deep Learning pipeline for animal image classification, structured into 6 phases.

## Setup and Kaggle Authentication

Ensure your `kaggle.json` credentials are set. For demonstration, we will set them using environment variables.

In [ ]:
import os

# Ensure Kaggle credentials are set as environment variables
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USERNAME"  # Replace with your Kaggle username
os.environ["KAGGLE_KEY"] = "YOUR_KAGGLE_KEY"  # Replace with your Kaggle API key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from skimage import io, color, exposure, feature, filters, measure, transform, segmentation
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
import seaborn as sns
import glob
import random
import shutil

plt.rcParams['figure.figsize'] = (10, 6)


## Phase 1: Image Collection and Discretization

**Data Loading:** Download a subset of images from the Kaggle dataset.
**Color Space Conversion:** Convert between RGB, HSV, and Grayscale.
**Discretization & Quantization:** Image sampling and reducing bit depth.
**Visualization & Histogram Analysis:** Plot example images with parameters and their intensity histograms.

In [ ]:
# 1.1 Data Loading
import kaggle
import zipfile

dataset_name = "iamsouravbanerjee/animal-image-dataset-90-different-animals"
download_path = "./dataset"

if not os.path.exists(download_path):
    print("Downloading dataset...")
    kaggle.api.dataset_download_cli(dataset_name, path=download_path, unzip=True)
    print("Dataset downloaded and extracted.")
else:
    print("Dataset already exists.")

# Select a subset of 10 random animal classes
data_dir = os.path.join(download_path, "animals", "animals") # Adjust based on actual zip structure
all_classes = os.listdir(data_dir)
random.seed(42)
selected_classes = random.sample(all_classes, 10)
print(f"Selected classes: {selected_classes}")

# Load paths for a few images from the first selected class to demonstrate Phase 1
sample_class = selected_classes[0]
sample_images_paths = glob.glob(os.path.join(data_dir, sample_class, "*.jpg"))[:3]

# Load first image for demonstration
img_path = sample_images_paths[0]
img_rgb = io.imread(img_path)


In [ ]:
# 1.2 Color Space Conversion
img_hsv = color.rgb2hsv(img_rgb)
img_gray = color.rgb2gray(img_rgb)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_rgb)
axes[0].set_title('Original (RGB)')
axes[0].axis('off')

axes[1].imshow(img_hsv)
axes[1].set_title('HSV Color Space')
axes[1].axis('off')

axes[2].imshow(img_gray, cmap='gray')
axes[2].set_title('Grayscale')
axes[2].axis('off')
plt.show()


In [ ]:
# 1.3 Discretization (Resampling) & Quantization (Bit Depth Reduction)
# Discretization: Downsample the image
scale_factor = 0.1
img_downsampled = transform.rescale(img_gray, scale_factor, anti_aliasing=True)

# Quantization: Reduce bit depth (e.g., from 8-bit to 3-bit / 8 levels)
num_levels = 8
img_quantized = np.floor(img_gray * num_levels) / (num_levels - 1)

# 1.4 Visualization and 1.5 Histogram Analysis
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original Grayscale
axes[0, 0].imshow(img_gray, cmap='gray')
axes[0, 0].set_title(f'Original Grayscale\nRes: {img_gray.shape}, Depth: 8-bit')
axes[0, 0].axis('off')

# Original Histogram
axes[1, 0].hist(img_gray.ravel(), bins=256, color='black', alpha=0.7)
axes[1, 0].set_title('Original Intensity Histogram')
axes[1, 0].set_xlabel('Pixel Intensity')
axes[1, 0].set_ylabel('Frequency')

# Quantized Image
axes[0, 1].imshow(img_quantized, cmap='gray')
axes[0, 1].set_title(f'Quantized\nRes: {img_gray.shape}, Depth: 3-bit')
axes[0, 1].axis('off')

# Quantized Histogram
axes[1, 1].hist(img_quantized.ravel(), bins=num_levels, color='black', alpha=0.7)
axes[1, 1].set_title('Quantized Intensity Histogram')
axes[1, 1].set_xlabel('Pixel Intensity')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Visualize Downsampled
plt.figure(figsize=(5,5))
plt.imshow(img_downsampled, cmap='gray')
plt.title(f'Downsampled (Discretization)\nRes: {img_downsampled.shape}')
plt.axis('off')
plt.show()


## Phase 2: Basic Image Processing Algorithms

**Filtering:** Compare Gaussian, Sobel, Laplacian, and Median filters.
**Geometric Transformations:** Apply rotation, scaling, and flipping.
**Thresholding:** Implement and compare global and adaptive thresholding.
**Analysis:** Text explanations on when each algorithm is best used.

In [ ]:
# 2.1 Filtering
# Apply filters to the grayscale image
img_gaussian = filters.gaussian(img_gray, sigma=1)
img_sobel = filters.sobel(img_gray)
img_laplace = filters.laplace(img_gray)
# Median filter requires image to be in specific formats (e.g., uint8)
from skimage.util import img_as_ubyte
img_median = filters.median(img_as_ubyte(img_gray))

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title('Original Grayscale')
axes[0].axis('off')

axes[1].imshow(img_gaussian, cmap='gray')
axes[1].set_title('Gaussian Filter (Blur)')
axes[1].axis('off')

axes[2].imshow(img_sobel, cmap='gray')
axes[2].set_title('Sobel Filter (Edge)')
axes[2].axis('off')

axes[3].imshow(img_laplace, cmap='gray')
axes[3].set_title('Laplacian Filter (Edge)')
axes[3].axis('off')

axes[4].imshow(img_median, cmap='gray')
axes[4].set_title('Median Filter (Noise Reduction)')
axes[4].axis('off')

plt.show()


In [ ]:
# 2.2 Geometric Transformations
# Rotation
img_rotated = transform.rotate(img_rgb, angle=45)

# Scaling (already partially demonstrated in Phase 1, but let's upscale)
img_scaled = transform.rescale(img_rgb, scale=1.5, channel_axis=-1, anti_aliasing=True)

# Flipping (Horizontal and Vertical)
img_flipped_h = np.fliplr(img_rgb)
img_flipped_v = np.flipud(img_rgb)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(img_rotated)
axes[0].set_title('Rotated 45 degrees')
axes[0].axis('off')

axes[1].imshow(img_scaled)
axes[1].set_title(f'Scaled (1.5x)\n{img_scaled.shape}')
axes[1].axis('off')

axes[2].imshow(img_flipped_h)
axes[2].set_title('Horizontal Flip')
axes[2].axis('off')

axes[3].imshow(img_flipped_v)
axes[3].set_title('Vertical Flip')
axes[3].axis('off')

plt.show()


In [ ]:
# 2.3 Thresholding
# Global Thresholding (Otsu's method)
global_thresh_val = filters.threshold_otsu(img_gray)
img_global_thresh = img_gray > global_thresh_val

# Adaptive Thresholding (Local)
block_size = 35
img_adaptive_thresh = filters.threshold_local(img_gray, block_size, offset=10)
img_adaptive_result = img_gray > img_adaptive_thresh

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_gray, cmap='gray')
axes[0].set_title('Original Grayscale')
axes[0].axis('off')

axes[1].imshow(img_global_thresh, cmap='gray')
axes[1].set_title(f'Global Thresholding (Otsu)\nThresh={global_thresh_val:.2f}')
axes[1].axis('off')

axes[2].imshow(img_adaptive_result, cmap='gray')
axes[2].set_title('Adaptive Local Thresholding')
axes[2].axis('off')

plt.show()


### 2.4 Analysis of Basic Image Processing Algorithms

**Filtering:**
- **Gaussian Filter:** Best used for smoothing an image and reducing general Gaussian noise. It acts as a low-pass filter, which is great as a preprocessing step before edge detection to avoid detecting false edges caused by noise.
- **Sobel Filter:** Best for detecting edges in a specific direction (horizontal/vertical) or overall gradient magnitude. Useful for feature extraction and finding structural outlines of animals.
- **Laplacian Filter:** Best for detecting fine details and sharp edges (zero-crossings of the second derivative). It is highly sensitive to noise, so it is usually applied after a Gaussian blur (Laplacian of Gaussian - LoG).
- **Median Filter:** Best for removing salt-and-pepper (impulse) noise while preserving sharp edges, unlike Gaussian blur which softens edges.

**Geometric Transformations:**
- Useful for **Data Augmentation** during the deep learning phase (Phase 5). Rotating, scaling, and flipping help the model generalize better by ensuring it isn't rotation or scale-dependent.

**Thresholding:**
- **Global Thresholding (Otsu):** Works perfectly when the image has a bimodal histogram (distinct foreground and background intensity distributions) and uniform lighting. 
- **Adaptive Thresholding:** Crucial when the image has varying lighting conditions (e.g., shadows over an animal). It calculates a threshold dynamically for small regions, successfully separating foreground from background even in uneven illumination.


## Phase 3: Measuring Object Parameters on Images

**Segmentation:** Segment objects within the images.
**Geometric Parameters:** Calculate surface area, perimeter, shape factor, and bounding box.
**Spatial Measurements:** Measure distances and angles between objects.
**Visualization & Reporting:** Draw results on images and output a formatted table for 10 objects.

In [ ]:
# 3.1 Segmentation and 3.2 Geometric Parameters
import math

# Using a combination of Gaussian blur, thresholding, and morphological operations to segment the animal
# In practice, animal segmentation is hard without Deep Learning, so this is a basic approach.
blurred = filters.gaussian(img_gray, sigma=2)
thresh_val = filters.threshold_otsu(blurred)
binary_mask = blurred > thresh_val

# Fill holes and clear borders to get cleaner objects
from scipy import ndimage as ndi
filled_mask = ndi.binary_fill_holes(binary_mask)
cleared_mask = segmentation.clear_border(filled_mask)

# Label the segmented objects
label_image = measure.label(cleared_mask)

# Extract region properties
regions = measure.regionprops(label_image)

# Filter out very small regions (noise)
filtered_regions = [r for r in regions if r.area > 500]

# Prepare data for reporting
measured_data = []

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(img_rgb)
ax.set_title('Segmented Objects and Parameters Overlay')

for idx, region in enumerate(filtered_regions[:10]): # Process up to 10 objects
    y0, x0 = region.centroid
    minr, minc, maxr, maxc = region.bbox
    
    area = region.area
    perimeter = region.perimeter
    # Shape factor (Circularity): 4 * pi * Area / Perimeter^2
    shape_factor = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
    
    # Store metrics
    measured_data.append({
        'Object ID': idx + 1,
        'Area': area,
        'Perimeter': round(perimeter, 2),
        'Shape Factor': round(shape_factor, 4),
        'Bounding Box': (minr, minc, maxr, maxc),
        'Centroid': (round(y0, 2), round(x0, 2))
    })
    
    # 3.4 Visualization (Overlay)
    rect = plt.Rectangle((minc, minr), maxc - minc, maxr - minr, fill=False, edgecolor='red', linewidth=2)
    ax.add_patch(rect)
    ax.plot(x0, y0, '.b', markersize=10) # Centroid
    ax.text(minc, minr - 5, f"ID: {idx+1}", color='yellow', fontsize=12, fontweight='bold', bbox=dict(facecolor='red', alpha=0.5))

plt.axis('off')
plt.show()

# 3.3 Spatial Measurements (Distance between the first two objects, if they exist)
if len(filtered_regions) >= 2:
    r1, r2 = filtered_regions[0], filtered_regions[1]
    y1, x1 = r1.centroid
    y2, x2 = r2.centroid
    
    distance = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    angle = math.degrees(math.atan2(y2 - y1, x2 - x1))
    
    print(f"Spatial Measurements between Object 1 and Object 2:")
    print(f"Distance: {distance:.2f} pixels")
    print(f"Angle: {angle:.2f} degrees")
else:
    print("Not enough objects segmented to measure spatial distance.")

# 3.5 Reporting
df_measurements = pd.DataFrame(measured_data)
print("\n--- Object Measurements Table ---")
display(df_measurements)


## Phase 4: Image Feature Extraction

**Texture Features:** Extract GLCM or LBP features.
**HOG Features:** Compute and visualize the Histogram of Oriented Gradients.
**Dimensionality Reduction:** Apply PCA/t-SNE and plot 2D/3D feature space.
**Classic ML:** Train and compare SVM and k-NN classifiers on extracted features.

In [ ]:
# Helper function to extract features for a dataset subset
def extract_features(image_paths):
    lbp_features = []
    hog_features = []
    labels = []
    
    # We will sample 10 images from each of the 10 selected classes to keep it fast
    for cls in selected_classes:
        cls_paths = glob.glob(os.path.join(data_dir, cls, "*.jpg"))[:10]
        for path in cls_paths:
            try:
                img = io.imread(path)
                if len(img.shape) == 3:
                    img_gray = color.rgb2gray(img)
                else:
                    img_gray = img
                
                img_resized = transform.resize(img_gray, (128, 128), anti_aliasing=True)
                
                # 4.1 Texture Features: Local Binary Pattern (LBP)
                radius = 3
                n_points = 8 * radius
                lbp = feature.local_binary_pattern(img_resized, n_points, radius, method='uniform')
                (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
                hist = hist.astype("float")
                hist /= (hist.sum() + 1e-7)
                lbp_features.append(hist)
                
                # 4.2 HOG Features
                fd = feature.hog(img_resized, orientations=9, pixels_per_cell=(16, 16),
                                 cells_per_block=(2, 2), visualize=False, feature_vector=True)
                hog_features.append(fd)
                
                labels.append(cls)
            except Exception as e:
                pass # Skip corrupted images
                
    return np.array(lbp_features), np.array(hog_features), np.array(labels)

print("Extracting features... This might take a minute.")
X_lbp, X_hog, y_labels = extract_features(sample_images_paths)
print(f"Extracted LBP shape: {X_lbp.shape}")
print(f"Extracted HOG shape: {X_hog.shape}")


In [ ]:
# Visualize HOG for the sample image
img_gray_resized = transform.resize(img_gray, (128, 128), anti_aliasing=True)
fd, hog_image = feature.hog(img_gray_resized, orientations=9, pixels_per_cell=(16, 16),
                            cells_per_block=(2, 2), visualize=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
ax1.axis('off')
ax1.imshow(img_gray_resized, cmap=plt.cm.gray)
ax1.set_title('Resized Input Image')

# Rescale histogram for better display
hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))
ax2.axis('off')
ax2.imshow(hog_image_rescaled, cmap=plt.cm.gray)
ax2.set_title('Histogram of Oriented Gradients (HOG)')
plt.show()


In [ ]:
# 4.3 Dimensionality Reduction
# Convert string labels to integers for plotting
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)

# Apply PCA on HOG features
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_hog)

# Apply t-SNE on HOG features
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
# t-SNE might struggle with very few samples relative to perplexity. Adjusting perplexity if needed.
n_samples = X_hog.shape[0]
if n_samples < 30:
    tsne = TSNE(n_components=2, perplexity=n_samples - 1, random_state=42)

X_tsne = tsne.fit_transform(X_hog)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot PCA
scatter = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y_encoded, cmap='tab10', alpha=0.7)
ax1.set_title('PCA on HOG Features (2D Space)')
ax1.set_xlabel('Principal Component 1')
ax1.set_ylabel('Principal Component 2')

# Plot t-SNE
scatter2 = ax2.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_encoded, cmap='tab10', alpha=0.7)
ax2.set_title('t-SNE on HOG Features (2D Space)')
ax2.set_xlabel('t-SNE Dimension 1')
ax2.set_ylabel('t-SNE Dimension 2')

plt.show()


In [ ]:
# 4.4 Classic ML Classification (SVM vs k-NN)
# Combine HOG and LBP for robust features
X_combined = np.hstack((X_lbp, X_hog))

X_train, X_test, y_train, y_test = train_test_split(X_combined, y_encoded, test_size=0.3, random_state=42)

# Train SVM
svm_clf = SVC(kernel='linear')
svm_clf.fit(X_train, y_train)
svm_preds = svm_clf.predict(X_test)
svm_acc = accuracy_score(y_test, svm_preds)

# Train k-NN
knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train, y_train)
knn_preds = knn_clf.predict(X_test)
knn_acc = accuracy_score(y_test, knn_preds)

print(f"SVM Accuracy (HOG + LBP): {svm_acc:.4f}")
print(f"k-NN Accuracy (HOG + LBP): {knn_acc:.4f}")

# Note: Accuracy might be low due to the small sample size (10 images/class) used for demonstration.


## Phase 5: Convolutional Neural Network (Classification)

**Data Prep:** Normalization, augmentation, and splitting (70/15/15).
**Custom CNN:** Design and build a custom architecture.
**Evaluation:** Accuracy, Precision, Recall, F1-score.
**Visualizations:** Confusion matrix and learning curves.

In [ ]:
# 5.1 Data Preparation and 5.2 Dataset Splitting
# Create dedicated directories for Train, Validation, and Test sets (70/15/15 split)
import shutil
import os
import random

base_subset_dir = "./dataset_split"
train_dir = os.path.join(base_subset_dir, "train")
val_dir = os.path.join(base_subset_dir, "val")
test_dir = os.path.join(base_subset_dir, "test")

# Clean up if exists
if os.path.exists(base_subset_dir):
    shutil.rmtree(base_subset_dir)

for d in [train_dir, val_dir, test_dir]:
    os.makedirs(d)

for cls in selected_classes:
    os.makedirs(os.path.join(train_dir, cls))
    os.makedirs(os.path.join(val_dir, cls))
    os.makedirs(os.path.join(test_dir, cls))
    
    # Get all images for the class
    src_dir = os.path.join(data_dir, cls)
    images = glob.glob(os.path.join(src_dir, "*.jpg"))
    random.shuffle(images)
    
    # Calculate splits (70/15/15)
    train_split = int(0.7 * len(images))
    val_split = int(0.85 * len(images)) # 70 + 15
    
    train_images = images[:train_split]
    val_images = images[train_split:val_split]
    test_images = images[val_split:]
    
    for img in train_images: shutil.copy(img, os.path.join(train_dir, cls))
    for img in val_images: shutil.copy(img, os.path.join(val_dir, cls))
    for img in test_images: shutil.copy(img, os.path.join(test_dir, cls))

print("Dataset successfully split into 70/15/15 Train/Validation/Test directories.")

# Data augmentation and normalization
batch_size = 32
img_height = 128
img_width = 128

train_datagen = ImageDataGenerator(
    rescale=1./255,           # Normalization
    rotation_range=20,        # Augmentation: Rotation
    width_shift_range=0.2,    # Augmentation: Shift
    height_shift_range=0.2,
    horizontal_flip=True,     # Augmentation: Flip
    brightness_range=[0.8, 1.2] # Augmentation: Brightness
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
# 5.3 Custom CNN Architecture
def build_custom_cnn(input_shape, num_classes):
    model = models.Sequential()
    
    # Block 1
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))
    
    # Block 2
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    
    # Block 3
    model.add(layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    
    # Dense Layers
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    return model

num_classes = len(selected_classes)
model_cnn = build_custom_cnn((img_height, img_width, 3), num_classes)
model_cnn.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

model_cnn.summary()


In [ ]:
# Train the model
epochs = 10 # Kept small for demonstration purposes
history = model_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs
)


In [ ]:
# 5.4 Evaluation Metrics and 5.5 Visualizations
# Learning Curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

# Plot Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.show()

# Predictions for Evaluation
test_generator.reset()
Y_pred = model_cnn.predict(test_generator)
y_pred_classes = np.argmax(Y_pred, axis=1)
y_true = test_generator.classes

# Calculate Metrics
acc = accuracy_score(y_true, y_pred_classes)
prec = precision_score(y_true, y_pred_classes, average='weighted', zero_division=0)
rec = recall_score(y_true, y_pred_classes, average='weighted', zero_division=0)
f1 = f1_score(y_true, y_pred_classes, average='weighted', zero_division=0)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1-Score: {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=test_generator.class_indices.keys(),
            yticklabels=test_generator.class_indices.keys())
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


## Phase 6: CNN Model Optimization and Advanced Techniques

**Architecture Comparison & Regularization:** Train a second CNN with Dropout and L2 regularization.
**Learning Rate Study:** Visualize the impact of different learning rates.
**Transfer Learning:** Fine-tune a pre-trained MobileNetV2 model.
**Explainable AI:** Grad-CAM visualizations.
**Localization:** Object detection using YOLO to predict bounding boxes.

In [ ]:
# 6.1 Architecture Comparison & 6.2 Regularization Study
def build_optimized_cnn(input_shape, num_classes):
    model = models.Sequential()
    
    # Block 1
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape,
                            kernel_regularizer=regularizers.l2(0.001))) # L2 Regularization
    model.add(layers.MaxPooling2D((2, 2)))
    
    # Block 2
    model.add(layers.Conv2D(64, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25)) # Dropout to mitigate overfitting
    
    # Block 3
    model.add(layers.Conv2D(128, (3, 3), activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))
    
    # Dense Layers
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    model.add(layers.Dropout(0.5)) # Heavy Dropout
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    return model

model_opt = build_optimized_cnn((img_height, img_width, 3), num_classes)
model_opt.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("Training Optimized CNN (with L2 and Dropout)...")
history_opt = model_opt.fit(train_generator, validation_data=val_generator, epochs=epochs, verbose=0)
print(f"Optimized CNN Val Accuracy: {history_opt.history['val_accuracy'][-1]:.4f}")


In [ ]:
# 6.3 Learning Rate Study
learning_rates = [0.01, 0.0001]
lr_histories = {}

for lr in learning_rates:
    print(f"Training with Learning Rate = {lr}...")
    model_lr = build_custom_cnn((img_height, img_width, 3), num_classes) # Use base model for comparison
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    model_lr.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    hist = model_lr.fit(train_generator, validation_data=val_generator, epochs=5, verbose=0) # Fewer epochs for speed
    lr_histories[lr] = hist.history['val_accuracy']

plt.figure(figsize=(8, 5))
for lr, acc in lr_histories.items():
    plt.plot(acc, label=f'LR = {lr}')
plt.plot(history.history['val_accuracy'][:5], label='LR = 0.001 (Default Adam)')
plt.title('Impact of Learning Rate on Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.show()


In [ ]:
# 6.4 Transfer Learning
base_model = MobileNetV2(input_shape=(img_height, img_width, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Freeze base model layers

tl_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

tl_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
                 loss='categorical_crossentropy', metrics=['accuracy'])

print("Training Transfer Learning Model (MobileNetV2)...")
history_tl = tl_model.fit(train_generator, validation_data=val_generator, epochs=epochs, verbose=0)
print(f"Transfer Learning Val Accuracy: {history_tl.history['val_accuracy'][-1]:.4f}")


In [ ]:
# 6.5 Explainable AI (Grad-CAM)
# Simple Grad-CAM implementation
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# Prepare an image for Grad-CAM
sample_img_path = sample_images_paths[0]
img_for_cam = tf.keras.preprocessing.image.load_img(sample_img_path, target_size=(img_height, img_width))
img_array = tf.keras.preprocessing.image.img_to_array(img_for_cam)
img_array = np.expand_dims(img_array, axis=0) / 255.0

# Using our custom CNN, get the name of the last conv layer
last_conv_layer = [layer.name for layer in model_cnn.layers if isinstance(layer, layers.Conv2D)][-1]

heatmap = make_gradcam_heatmap(img_array, model_cnn, last_conv_layer)

# Overlay heatmap
heatmap_resized = cv2.resize(heatmap, (img_width, img_height))
heatmap_resized = np.uint8(255 * heatmap_resized)
heatmap_col = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)
superimposed_img = heatmap_col * 0.4 + img_array[0] * 255

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_for_cam)
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(superimposed_img.astype('uint8'))
plt.title('Grad-CAM Heatmap')
plt.axis('off')
plt.show()


In [ ]:
# 6.6 Localization (Bounding Box Regression)
from ultralytics import YOLO
import cv2

# 1. Generate Pseudo-Ground Truth using YOLO
print("Generating pseudo-ground truth bounding boxes using YOLOv8n...")
yolo_model = YOLO('yolov8n.pt')

bbox_data = []
# We'll use the subset of images we already collected paths for in Phase 4 (sample_images_paths can be extended to all training images)
# For demonstration, let's process 50 images from our training subset
all_subset_paths = []
for cls in selected_classes:
    all_subset_paths.extend(glob.glob(os.path.join(base_subset_dir, cls, "*.jpg")))

# Limit to a small number for runtime demonstration
demo_paths = all_subset_paths[:50] 

X_loc = []
y_loc = [] # Target format: [x_min, y_min, width, height] normalized

for path in demo_paths:
    try:
        img = cv2.imread(path)
        if img is None: continue
        h, w, _ = img.shape
        
        # Run YOLO
        results = yolo_model(path, verbose=False)
        result = results[0]
        
        # If YOLO detects an object, take the first one (most prominent)
        if len(result.boxes.xywh) > 0:
            # ultralytics xywh is center_x, center_y, width, height
            # we need to normalize these coordinates between 0 and 1
            box = result.boxes.xywh[0].cpu().numpy()
            cx, cy, bw, bh = box
            
            # Convert center-based to top-left based [x, y, w, h] normalized
            x_min_norm = (cx - bw/2) / w
            y_min_norm = (cy - bh/2) / h
            w_norm = bw / w
            h_norm = bh / h
            
            # Prepare image for our CNN
            img_resized = cv2.resize(img, (img_height, img_width))
            img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
            
            X_loc.append(img_rgb / 255.0)
            y_loc.append([x_min_norm, y_min_norm, w_norm, h_norm])
    except Exception as e:
        pass

X_loc = np.array(X_loc)
y_loc = np.array(y_loc)

print(f"Generated bounding box targets for {len(X_loc)} images.")

if len(X_loc) > 0:
    # Split for localization training
    X_loc_train, X_loc_test, y_loc_train, y_loc_test = train_test_split(X_loc, y_loc, test_size=0.2, random_state=42)

    # 2. Add a Regression Head to our CNN for Localization
    loc_input = layers.Input(shape=(img_height, img_width, 3))
    
    # We can reuse the feature extractor from our optimized CNN (or build a simple one)
    x = layers.Conv2D(32, (3, 3), activation='relu')(loc_input)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    
    # Regression Head (4 units: x, y, w, h)
    bbox_output = layers.Dense(4, activation='sigmoid', name='bbox_output')(x) # Sigmoid because coordinates are normalized [0,1]
    
    localization_model = models.Model(inputs=loc_input, outputs=bbox_output)
    localization_model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    
    print("Training Localization Regression Model...")
    # Train for a few epochs
    localization_model.fit(X_loc_train, y_loc_train, validation_data=(X_loc_test, y_loc_test), epochs=10, batch_size=8, verbose=0)
    
    # 3. Evaluate Predictions using Intersection over Union (IoU)
    def calculate_iou(boxA, boxB):
        # Determine the (x, y)-coordinates of the intersection rectangle
        # box format: [x, y, w, h] (normalized)
        xA = max(boxA[0], boxB[0])
        yA = max(boxA[1], boxB[1])
        xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
        yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])

        # Compute the area of intersection rectangle
        interArea = max(0, xB - xA) * max(0, yB - yA)

        # Compute the area of both the prediction and ground-truth rectangles
        boxAArea = boxA[2] * boxA[3]
        boxBArea = boxB[2] * boxB[3]

        # Compute IoU
        iou = interArea / float(boxAArea + boxBArea - interArea + 1e-6)
        return iou

    print("Evaluating with IoU metric on Test Set...")
    preds = localization_model.predict(X_loc_test)
    ious = [calculate_iou(true_box, pred_box) for true_box, pred_box in zip(y_loc_test, preds)]
    mean_iou = np.mean(ious)
    print(f"Mean IoU for Localization Head: {mean_iou:.4f}")
    
    # Visualize a prediction vs ground truth
    idx = 0
    test_img = X_loc_test[idx]
    pred_box = preds[idx]
    true_box = y_loc_test[idx]
    
    # Convert normalized back to absolute pixels for drawing
    true_x, true_y, true_w, true_h = [int(val * img_width) if i%2==0 else int(val * img_height) for i, val in enumerate(true_box)]
    pred_x, pred_y, pred_w, pred_h = [int(val * img_width) if i%2==0 else int(val * img_height) for i, val in enumerate(pred_box)]
    
    img_draw = np.copy(test_img)
    # Ground Truth = Green
    cv2.rectangle(img_draw, (true_x, true_y), (true_x + true_w, true_y + true_h), (0, 1, 0), 2)
    # Prediction = Red
    cv2.rectangle(img_draw, (pred_x, pred_y), (pred_x + pred_w, pred_y + pred_h), (1, 0, 0), 2)
    
    plt.figure(figsize=(6,6))
    plt.imshow(img_draw)
    plt.title(f'Localization\nGreen=True, Red=Pred\nIoU={ious[idx]:.2f}')
    plt.axis('off')
    plt.show()

else:
    print("Could not generate sufficient bounding box data for training.")
